# 🤖 트랜스포머(Transformer) 이론 실습
### [Deep Learning 101] 트랜스포머, 스텝 바이 스텝 | 신박AI

---

**학습 목표**
- 트랜스포머의 각 구성요소를 수식 없이 직접 계산하며 이해한다
- 인코더(Encoder) / 디코더(Decoder) 구조를 단계별로 실습한다
- Attention 메커니즘을 직접 행렬 연산으로 구현한다

**예제 번역 태스크**  
> 영어 → 한국어: `"I am fine"` → `"나는 괜찮아"`


## 📦 0. 라이브러리 임포트

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
matplotlib.rcParams['axes.unicode_minus'] = False
try:
    plt.rcParams['font.family'] = 'NanumGothic'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

np.set_printoptions(precision=4, suppress=True)
print("✅ 라이브러리 로드 완료")

---
## 📖 1. 어휘 사전(Vocabulary) & 토크나이징

트랜스포머는 텍스트를 바로 처리하지 못합니다.  
먼저 단어를 **숫자(인덱스)** 로 바꾸는 작업이 필요합니다.

| 토큰 | 인덱스 |
|------|--------|
| `<SOS>` | 0 |
| `<EOS>` | 1 |
| `<PAD>` | 2 |
| `I` | 3 |
| `am` | 4 |
| `fine` | 5 |
| `나는` | 6 |
| `괜찮아` | 7 |

In [ ]:
# 어휘 사전 정의
vocab = {
    '<SOS>': 0,  # Start of Sentence
    '<EOS>': 1,  # End of Sentence
    '<PAD>': 2,  # Padding
    'I': 3,
    'am': 4,
    'fine': 5,
    '나는': 6,
    '괜찮아': 7
}
idx2word = {v: k for k, v in vocab.items()}
VOCAB_SIZE = len(vocab)  # 8

# 입력 문장 토크나이징
src_sentence = ['I', 'am', 'fine']
tgt_sentence = ['나는', '괜찮아']

# 인덱스로 변환
src_tokens = [vocab[w] for w in src_sentence]
# 디코더 입력: <SOS> + 정답 (Teacher Forcing)
tgt_input  = [vocab['<SOS>']] + [vocab[w] for w in tgt_sentence]
# 디코더 정답 (Loss 계산용)
tgt_output = [vocab[w] for w in tgt_sentence] + [vocab['<EOS>']]

print(f"📌 소스 문장   : {src_sentence}")
print(f"📌 소스 토큰   : {src_tokens}")
print(f"📌 디코더 입력 : {tgt_input}  ({[idx2word[i] for i in tgt_input]})")
print(f"📌 디코더 정답 : {tgt_output} ({[idx2word[i] for i in tgt_output]})")

---
## 🔢 2. 임베딩(Embedding) + 위치 인코딩(Positional Encoding)

### 2-1. 임베딩
각 토큰 인덱스를 `d_model` 차원의 벡터로 변환합니다.  
쉽게 말하면 **단어 → 실수 벡터** 룩업 테이블입니다.

```
토큰 인덱스 → [Embedding Table] → 벡터
     3      →     [0.2, -0.1, 0.8, ...]  (d_model 차원)
```

### 2-2. 위치 인코딩
트랜스포머는 순서 정보가 없으므로, **위치 정보를 임베딩에 더해줍니다.**

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
# 하이퍼파라미터
d_model = 8   # 임베딩 차원 (실제는 512)
np.random.seed(42)

# 임베딩 테이블 초기화 (vocab_size x d_model)
embedding_table = np.random.randn(VOCAB_SIZE, d_model) * 0.1

def get_embedding(token_ids):
    """토큰 인덱스 리스트를 임베딩 행렬로 변환"""
    return embedding_table[token_ids]  # (seq_len, d_model)

def positional_encoding(seq_len, d_model):
    """사인/코사인 위치 인코딩"""
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(d_model // 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, 2*i]   = np.sin(angle)
            PE[pos, 2*i+1] = np.cos(angle)
    return PE

# 소스 임베딩 + 위치 인코딩
src_emb = get_embedding(src_tokens)                         # (3, 8)
src_PE  = positional_encoding(len(src_tokens), d_model)     # (3, 8)
src_X   = src_emb + src_PE                                  # (3, 8)

# 타겟 임베딩 + 위치 인코딩
tgt_emb = get_embedding(tgt_input)                          # (3, 8)
tgt_PE  = positional_encoding(len(tgt_input), d_model)      # (3, 8)
tgt_X   = tgt_emb + tgt_PE                                  # (3, 8)

print(f"📐 소스 임베딩 형태: {src_emb.shape}")
print(f"📐 위치 인코딩 형태: {src_PE.shape}")
print(f"📐 인코더 입력(X)   : {src_X.shape}\n")
print("소스 입력 행렬 X (인코더 입력):")
print(src_X)

# 위치 인코딩 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
PE_vis = positional_encoding(20, d_model)
im = axes[0].imshow(PE_vis, cmap='RdBu', aspect='auto')
axes[0].set_title('위치 인코딩 시각화 (20 positions)', fontsize=12)
axes[0].set_xlabel('차원 (d_model)')
axes[0].set_ylabel('위치 (position)')
plt.colorbar(im, ax=axes[0])

for i in range(d_model):
    axes[1].plot(PE_vis[:, i], alpha=0.6, label=f'dim {i}')
axes[1].set_title('차원별 위치 인코딩 파형', fontsize=12)
axes[1].set_xlabel('위치')
axes[1].set_ylabel('값')
axes[1].legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()
print("✅ 위치 인코딩 완료: 짝수 차원=sin, 홀수 차원=cos")

---
## 🎯 3. Self-Attention (자기 어텐션) - 핵심!

트랜스포머의 심장부입니다. **"각 단어가 다른 단어들과 얼마나 관련있는가"** 를 계산합니다.

### Q, K, V 란?
| 이름 | 의미 | 역할 |
|------|------|------|
| **Q** (Query)  | 질문 | 내가 무엇을 찾고 있는가? |
| **K** (Key)    | 열쇠 | 나는 어떤 정보를 가지고 있는가? |
| **V** (Value)  | 값   | 실제로 전달할 정보 |

### 계산 순서
```
1. Q = X @ W_Q
2. K = X @ W_K  
3. V = X @ W_V
4. Score = Q @ K.T / sqrt(d_k)   ← Scaled Dot-Product
5. Attention Weight = softmax(Score)
6. Output = Attention Weight @ V
```

In [ ]:
# Attention 차원
d_k = 4  # Query/Key 차원 (실제는 64)
d_v = 4  # Value 차원

# W_Q, W_K, W_V 가중치 행렬 초기화 (d_model x d_k)
np.random.seed(7)
W_Q = np.random.randn(d_model, d_k) * 0.1  # (8, 4)
W_K = np.random.randn(d_model, d_k) * 0.1  # (8, 4)
W_V = np.random.randn(d_model, d_v) * 0.1  # (8, 4)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    Q: (seq_q, d_k)
    K: (seq_k, d_k)
    V: (seq_k, d_v)
    """
    d_k = Q.shape[-1]
    
    # ① Score 계산
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_q, seq_k)
    print(f"  ① Score = Q @ K.T / √{d_k}:")
    print(f"     형태: {scores.shape}")
    print(f"     값:\n{scores}\n")
    
    # ② 마스킹 (선택적)
    if mask is not None:
        scores = scores + mask * (-1e9)  # -inf로 마스킹
        print(f"  ② 마스킹 적용 후:\n{scores}\n")
    
    # ③ Softmax → Attention Weight
    def softmax(x):
        e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return e_x / e_x.sum(axis=-1, keepdims=True)
    
    attn_weights = softmax(scores)     # (seq_q, seq_k)
    print(f"  ③ Attention Weight = softmax(Score):")
    print(f"     값:\n{attn_weights}")
    print(f"     행 합계(각 행의 합=1): {attn_weights.sum(axis=-1).round(4)}\n")
    
    # ④ Output = Attention Weight @ V
    output = attn_weights @ V          # (seq_q, d_v)
    print(f"  ④ Output = Attention Weight @ V:")
    print(f"     형태: {output.shape}")
    print(f"     값:\n{output}")
    
    return output, attn_weights

# ====== 인코더 Self-Attention 계산 ======
print("="*55)
print("🔵 인코더 Self-Attention 계산")
print("="*55)
print(f"입력 X 형태: {src_X.shape} (seq_len=3, d_model=8)\n")

Q = src_X @ W_Q   # (3, 4)
K = src_X @ W_K   # (3, 4)
V = src_X @ W_V   # (3, 4)

print(f"Q = X @ W_Q → 형태: {Q.shape}")
print(f"K = X @ W_K → 형태: {K.shape}")
print(f"V = X @ W_V → 형태: {V.shape}\n")

enc_attn_output, enc_attn_weights = scaled_dot_product_attention(Q, K, V)

In [ ]:
# ====== Attention 가중치 시각화 ======
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(enc_attn_weights, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)

tokens = src_sentence
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=13)
ax.set_yticklabels(tokens, fontsize=13)
ax.set_xlabel('Key (어떤 단어를 참조?)', fontsize=11)
ax.set_ylabel('Query (어떤 단어의 관점?)', fontsize=11)
ax.set_title('Encoder Self-Attention 가중치 히트맵', fontsize=13)

for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{enc_attn_weights[i,j]:.2f}',
                ha='center', va='center',
                color='white' if enc_attn_weights[i,j] > 0.5 else 'black',
                fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print("\n💡 행렬 읽는 법: 행=Query 단어, 열=Key 단어")
print("   값이 클수록(진할수록) 해당 Query가 그 Key에 더 많이 집중!")

---
## 🏗️ 4. Multi-Head Attention

Self-Attention을 여러 번 병렬로 수행하여 **다양한 관점**에서 문맥을 파악합니다.

```
Head 1: 문법적 관계에 집중
Head 2: 의미적 관계에 집중
Head 3: 위치적 관계에 집중
        ...
→ 모든 Head 출력을 Concat → Linear 변환
```

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

In [ ]:
def multi_head_attention(X_q, X_k, X_v, num_heads=2, mask=None, verbose=False):
    """
    Multi-Head Attention
    X_q, X_k, X_v: (seq_len, d_model)
    """
    d_head = d_model // num_heads  # 각 헤드의 차원
    head_outputs = []
    all_attn_weights = []
    
    for h in range(num_heads):
        # 각 헤드별 독립적인 W_Q, W_K, W_V
        np.random.seed(h + 10)
        Wq_h = np.random.randn(d_model, d_head) * 0.1
        Wk_h = np.random.randn(d_model, d_head) * 0.1
        Wv_h = np.random.randn(d_model, d_head) * 0.1
        
        Qh = X_q @ Wq_h   # (seq_q, d_head)
        Kh = X_k @ Wk_h   # (seq_k, d_head)
        Vh = X_v @ Wv_h   # (seq_k, d_head)
        
        # Scaled Dot-Product Attention (조용히 계산)
        d_k = Qh.shape[-1]
        scores = Qh @ Kh.T / np.sqrt(d_k)
        if mask is not None:
            scores = scores + mask * (-1e9)
        e_x = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attn_w = e_x / e_x.sum(axis=-1, keepdims=True)
        out_h = attn_w @ Vh
        
        head_outputs.append(out_h)
        all_attn_weights.append(attn_w)
        
        if verbose:
            print(f"  Head {h+1}: 어텐션 가중치 형태={attn_w.shape}, 출력 형태={out_h.shape}")
    
    # Concat
    concat_out = np.concatenate(head_outputs, axis=-1)  # (seq_q, d_model)
    
    # Output Linear 변환
    np.random.seed(99)
    W_O = np.random.randn(d_model, d_model) * 0.1
    output = concat_out @ W_O  # (seq_q, d_model)
    
    if verbose:
        print(f"  Concat 후: {concat_out.shape}")
        print(f"  Linear 후: {output.shape}")
    
    return output, all_attn_weights

# 인코더 Multi-Head Attention 실행
num_heads = 2
print(f"{'='*55}")
print(f"🔵 Multi-Head Attention (head={num_heads})")
print(f"{'='*55}")
print(f"각 헤드 차원: d_model({d_model}) / num_heads({num_heads}) = {d_model//num_heads}\n")

mha_output, all_weights = multi_head_attention(src_X, src_X, src_X, num_heads=num_heads, verbose=True)
print(f"\n✅ 최종 Multi-Head Attention 출력: {mha_output.shape}")

# 헤드별 어텐션 시각화
fig, axes = plt.subplots(1, num_heads, figsize=(10, 4))
for h, (ax, weights) in enumerate(zip(axes, all_weights)):
    im = ax.imshow(weights, cmap='Oranges', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(src_sentence)))
    ax.set_yticks(range(len(src_sentence)))
    ax.set_xticklabels(src_sentence)
    ax.set_yticklabels(src_sentence)
    ax.set_title(f'Head {h+1} Attention', fontsize=12)
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            ax.text(j, i, f'{weights[i,j]:.2f}',
                    ha='center', va='center',
                    color='white' if weights[i,j] > 0.5 else 'black', fontsize=11)
plt.suptitle('Multi-Head Attention: 헤드별 어텐션 패턴', fontsize=13)
plt.tight_layout()
plt.show()

---
## ➕ 5. Add & Norm (잔차 연결 + 레이어 정규화)

### Residual Connection (잔차 연결)
입력을 출력에 더해주어 **기울기 소실 문제를 방지**합니다.
```
output = LayerNorm(X + SubLayer(X))
```

### Layer Normalization
각 샘플(행)의 특징 차원에 대해 정규화합니다.
$$\hat{x} = \frac{x - \mu}{\sigma + \epsilon}, \quad y = \gamma \hat{x} + \beta$$

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer Normalization: 각 행(토큰)별로 정규화"""
    mean = x.mean(axis=-1, keepdims=True)    # (seq_len, 1)
    std  = x.std(axis=-1, keepdims=True)     # (seq_len, 1)
    gamma = np.ones_like(x)   # 학습 파라미터 (초기값=1)
    beta  = np.zeros_like(x)  # 학습 파라미터 (초기값=0)
    return gamma * (x - mean) / (std + eps) + beta

def add_and_norm(x, sublayer_output):
    """Add & Norm: 잔차 연결 + 레이어 정규화"""
    return layer_norm(x + sublayer_output)

# 적용
enc_after_attn = add_and_norm(src_X, mha_output)  # (3, 8)

print("📊 Add & Norm 전후 비교")
print(f"  입력(X):         mean={src_X.mean():.4f}, std={src_X.std():.4f}")
print(f"  MHA 출력:        mean={mha_output.mean():.4f}, std={mha_output.std():.4f}")
print(f"  X + MHA:         mean={(src_X+mha_output).mean():.4f}, std={(src_X+mha_output).std():.4f}")
print(f"  LayerNorm 후:    mean={enc_after_attn.mean():.4f}, std={enc_after_attn.std():.4f}")
print(f"\n✅ LayerNorm 결과 형태: {enc_after_attn.shape}")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, data, title in zip(axes,
    [src_X, src_X + mha_output, enc_after_attn],
    ['입력 X', 'X + MHA (Add)', 'LayerNorm 후 (Add&Norm)']):
    im = ax.imshow(data, cmap='coolwarm', aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('차원')
    ax.set_ylabel('토큰')
    ax.set_yticks(range(3))
    ax.set_yticklabels(src_sentence)
    plt.colorbar(im, ax=ax)
plt.suptitle('Add & Norm 단계별 변화', fontsize=13)
plt.tight_layout()
plt.show()

---
## ⚡ 6. Feed-Forward Network (FFN)

각 위치(토큰)에 **독립적으로** 적용되는 2층 신경망입니다.

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2

차원 변화: d_model → d_ff → d_model
           8      → 16  → 8
(실제:     512    → 2048 → 512)
```

> 💡 Attention이 **토큰 간** 관계를 학습한다면,  
> FFN은 **각 토큰의 특징**을 더 풍부하게 변환합니다.

In [ ]:
d_ff = 16  # FFN 내부 차원 (실제는 2048)

# FFN 가중치
np.random.seed(20)
W1 = np.random.randn(d_model, d_ff) * 0.1  # (8, 16)
b1 = np.zeros(d_ff)                         # (16,)
W2 = np.random.randn(d_ff, d_model) * 0.1  # (16, 8)
b2 = np.zeros(d_model)                      # (8,)

def feed_forward(x):
    """Position-wise Feed-Forward Network"""
    hidden = np.maximum(0, x @ W1 + b1)  # ReLU 활성화
    output = hidden @ W2 + b2
    return output, hidden

ffn_out, ffn_hidden = feed_forward(enc_after_attn)

# Add & Norm 적용
enc_output = add_and_norm(enc_after_attn, ffn_out)  # (3, 8)

print("📊 FFN 계산 과정")
print(f"  입력:        {enc_after_attn.shape}")
print(f"  W1 곱 후:    {(enc_after_attn @ W1).shape}  → d_ff={d_ff}으로 확장")
print(f"  ReLU 후:     {ffn_hidden.shape}")
print(f"  W2 곱 후:    {ffn_out.shape}     → d_model={d_model}으로 복원")
print(f"  Add&Norm 후: {enc_output.shape}")
print(f"\n✅ 인코더 출력 완성!")
print(f"\n인코더 최종 출력 (3x8 행렬):")
print(enc_output)

---
## 🏛️ 7. 인코더 전체 구조 정리

지금까지 한 계산을 하나의 인코더 레이어 함수로 통합합니다.

```
입력 X
  ↓  (Embedding + Positional Encoding)
  ↓  [Multi-Head Self-Attention]
  ↓  [Add & Norm]
  ↓  [Feed-Forward Network]
  ↓  [Add & Norm]
인코더 출력 (Context)
```

In [ ]:
def encoder_layer(X, num_heads=2):
    """Transformer Encoder Layer 전체"""
    # 1. Multi-Head Self-Attention
    mha_out, _ = multi_head_attention(X, X, X, num_heads=num_heads)
    # 2. Add & Norm
    X = add_and_norm(X, mha_out)
    # 3. Feed-Forward Network
    ffn_out, _ = feed_forward(X)
    # 4. Add & Norm
    X = add_and_norm(X, ffn_out)
    return X

# 인코더 레이어 N번 쌓기 (실제 논문: N=6)
N = 2  # 레이어 수
encoder_context = src_X.copy()

print(f"🏗️ 인코더 {N}개 레이어 실행")
for layer_idx in range(N):
    encoder_context = encoder_layer(encoder_context)
    print(f"  Layer {layer_idx+1} 완료: {encoder_context.shape}")

print(f"\n✅ 인코더 최종 컨텍스트(Context):")
print(f"   형태: {encoder_context.shape}  (src_seq_len=3, d_model=8)")
print(encoder_context)
print("\n💡 이 Context 행렬이 디코더로 전달됩니다!")

# 인코더 구조 다이어그램
fig, ax = plt.subplots(figsize=(7, 8))
ax.axis('off')
components = [
    (0.5, 0.92, '입력 토큰\n[I, am, fine]', '#E8F5E9', '↓ 임베딩 + 위치 인코딩'),
    (0.5, 0.75, 'Multi-Head\nSelf-Attention', '#BBDEFB', '↓ Add & Norm'),
    (0.5, 0.58, 'Feed-Forward\nNetwork', '#FFF9C4', '↓ Add & Norm'),
    (0.5, 0.40, f'× {N} (N개 레이어)', '#F5F5F5', ''),
    (0.5, 0.23, '인코더 출력\n(Context Matrix)', '#C8E6C9', ''),
]
for x, y, text, color, below_text in components:
    ax.add_patch(plt.FancyBboxPatch((x-0.28, y-0.06), 0.56, 0.10,
                                     boxstyle="round,pad=0.01",
                                     facecolor=color, edgecolor='gray', linewidth=1.5,
                                     transform=ax.transAxes))
    ax.text(x, y-0.005, text, ha='center', va='center',
            fontsize=12, fontweight='bold', transform=ax.transAxes)
    if below_text:
        ax.annotate('', xy=(x, y-0.10), xytext=(x, y-0.06),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='dimgray', lw=2))
        ax.text(x+0.05, y-0.083, below_text, ha='left', va='center',
                fontsize=9, color='gray', transform=ax.transAxes)

ax.set_title('Transformer Encoder 구조', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## 🔓 8. 디코더(Decoder) - Masked Self-Attention

디코더는 인코더와 거의 같지만, **두 가지 차이**가 있습니다.

1. **Masked Self-Attention**: 미래 토큰을 볼 수 없도록 마스킹
2. **Cross-Attention**: 인코더의 출력(Context)을 참조

### Causal Mask (인과 마스크)
```
      <SOS> 나는  괜찮아
<SOS> [ 1    0     0  ]  ← <SOS>는 자신만 볼 수 있음
나는  [ 1    1     0  ]  ← '나는'은 <SOS>와 자신만
괜찮아[ 1    1     1  ]  ← '괜찮아'는 모두 볼 수 있음
```

In [ ]:
tgt_seq_len = len(tgt_input)  # 3 (<SOS>, 나는, 괜찮아)

def create_causal_mask(seq_len):
    """미래 토큰을 마스킹하는 하삼각 마스크 (0=볼수있음, 1=마스킹)"""
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # 상삼각을 1로
    return mask

causal_mask = create_causal_mask(tgt_seq_len)

print("🎭 Causal Mask (1=마스킹, 미래 참조 불가):")
tgt_tokens_display = [idx2word[i] for i in tgt_input]
print(f"     {'  '.join(tgt_tokens_display)}")
for i, row in enumerate(causal_mask):
    print(f"{tgt_tokens_display[i]:5s}: {row}")

# 타겟 임베딩 + 위치 인코딩
tgt_seq_input = tgt_X  # (3, 8)

# 디코더 Masked Self-Attention
def masked_self_attention_output(X, mask, num_heads=2):
    mha_out, weights = multi_head_attention(X, X, X, num_heads=num_heads, mask=mask)
    return add_and_norm(X, mha_out), weights

dec_after_masked_attn, masked_weights = masked_self_attention_output(tgt_seq_input, causal_mask)

# 마스킹된 어텐션 가중치 시각화
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, weights, title in zip(axes,
    [masked_weights[0], masked_weights[1]],
    ['Head 1 (Masked)', 'Head 2 (Masked)']):
    im = ax.imshow(weights, cmap='Purples', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(tgt_seq_len))
    ax.set_yticks(range(tgt_seq_len))
    ax.set_xticklabels(tgt_tokens_display)
    ax.set_yticklabels(tgt_tokens_display)
    ax.set_title(f'디코더 {title}', fontsize=11)
    for i in range(tgt_seq_len):
        for j in range(tgt_seq_len):
            ax.text(j, i, f'{weights[i,j]:.2f}',
                    ha='center', va='center',
                    color='white' if weights[i,j] > 0.5 else 'black', fontsize=11)
plt.suptitle('Masked Self-Attention: 미래 토큰 참조 불가', fontsize=12)
plt.tight_layout()
plt.show()
print("✅ 대각선 아래(하삼각)만 값 존재 → 미래 정보 차단!")

---
## 🔗 9. Cross-Attention (교차 어텐션)

디코더가 인코더의 출력을 참조하는 핵심 단계입니다!

```
Q → 디코더에서 온다  (내가 지금 생성하려는 단어)
K → 인코더에서 온다  (원문의 어떤 부분에 집중?)
V → 인코더에서 온다  (원문에서 가져올 정보)
```

> 💡 번역 예시: 한국어 '나는'을 생성할 때,  
> 영어 원문의 'I'에 높은 어텐션을 갖게 됩니다.

In [ ]:
# Cross-Attention: Q=디코더, K=V=인코더
cross_attn_out, cross_weights = multi_head_attention(
    X_q=dec_after_masked_attn,   # Query: 디코더
    X_k=encoder_context,          # Key:   인코더 출력
    X_v=encoder_context,          # Value: 인코더 출력
    num_heads=num_heads
)
dec_after_cross_attn = add_and_norm(dec_after_masked_attn, cross_attn_out)

print("📐 Cross-Attention 형태 확인:")
print(f"  Q (디코더): {dec_after_masked_attn.shape}  (타겟 seq_len={tgt_seq_len})")
print(f"  K (인코더): {encoder_context.shape}         (소스 seq_len=3)")
print(f"  V (인코더): {encoder_context.shape}")
print(f"  출력:       {cross_attn_out.shape}           (타겟 seq_len 유지)")

# Cross-Attention 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, weights, title in zip(axes, cross_weights, ['Head 1', 'Head 2']):
    im = ax.imshow(weights, cmap='YlOrRd', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(src_sentence)))
    ax.set_yticks(range(tgt_seq_len))
    ax.set_xticklabels(src_sentence, fontsize=12)
    ax.set_yticklabels(tgt_tokens_display, fontsize=12)
    ax.set_xlabel('Key: 인코더(원문)', fontsize=11)
    ax.set_ylabel('Query: 디코더(번역)', fontsize=11)
    ax.set_title(f'Cross-Attention {title}', fontsize=12)
    for i in range(tgt_seq_len):
        for j in range(len(src_sentence)):
            ax.text(j, i, f'{weights[i,j]:.2f}',
                    ha='center', va='center',
                    color='white' if weights[i,j] > 0.5 else 'black', fontsize=11)
plt.suptitle('Cross-Attention: 번역 토큰이 원문 어디를 참조하는가?', fontsize=12)
plt.tight_layout()
plt.show()

---
## 📊 10. 디코더 FFN → Linear → Softmax → 예측

디코더의 마지막 단계입니다.

```
디코더 출력 (seq_len, d_model)
     ↓  FFN + Add&Norm
     ↓  Linear (d_model → vocab_size)
     ↓  Softmax
각 위치에서 다음 단어 확률 분포!
```

In [ ]:
# 디코더 FFN + Add&Norm
dec_ffn_out, _ = feed_forward(dec_after_cross_attn)
dec_output = add_and_norm(dec_after_cross_attn, dec_ffn_out)  # (3, 8)

# Linear 레이어: d_model → vocab_size
np.random.seed(55)
W_linear = np.random.randn(d_model, VOCAB_SIZE) * 0.1  # (8, 8)
b_linear = np.zeros(VOCAB_SIZE)                         # (8,)
logits = dec_output @ W_linear + b_linear               # (3, 8)

# Softmax → 확률 분포
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

probs = softmax(logits)  # (3, 8)

print("📊 각 위치별 단어 확률 분포")
vocab_list = list(vocab.keys())
print(f"{'토큰위치':<10} {'예측 단어':<12} {'확률':<8} | 전체 분포")
print("-" * 60)
for pos, (token_name, prob_row) in enumerate(zip(tgt_tokens_display, probs)):
    pred_idx = np.argmax(prob_row)
    pred_word = vocab_list[pred_idx]
    pred_prob = prob_row[pred_idx]
    bar = '█' * int(pred_prob * 30)
    print(f"{token_name:<10} → {pred_word:<12} {pred_prob:.4f}  {bar}")

# 확률 분포 시각화
fig, axes = plt.subplots(1, tgt_seq_len, figsize=(14, 4), sharey=False)
colors = ['#4FC3F7', '#81C784', '#FFB74D']
for pos, (ax, token_name) in enumerate(zip(axes, tgt_tokens_display)):
    prob_row = probs[pos]
    bars = ax.bar(range(VOCAB_SIZE), prob_row, color=colors[pos], alpha=0.8, edgecolor='gray')
    # 최고 확률 강조
    best = np.argmax(prob_row)
    bars[best].set_edgecolor('red')
    bars[best].set_linewidth(2)
    ax.set_xticks(range(VOCAB_SIZE))
    ax.set_xticklabels(vocab_list, rotation=45, ha='right', fontsize=10)
    ax.set_title(f'위치 {pos}: {token_name} 입력 후\n예측={vocab_list[np.argmax(prob_row)]}', fontsize=11)
    ax.set_ylabel('확률')
    ax.set_ylim(0, 0.4)
plt.suptitle('디코더 출력: 각 위치의 다음 단어 확률 분포', fontsize=13)
plt.tight_layout()
plt.show()

---
## 📉 11. Loss 계산 (Cross-Entropy Loss)

모델의 예측이 정답과 얼마나 다른지 계산합니다.

```
디코더 입력: [<SOS>, 나는,   괜찮아  ]
정답 레이블: [나는,   괜찮아, <EOS>  ]  ← 한 칸씩 shift
```

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T}\log P(y_t \mid y_{<t}, X)$$

In [ ]:
def cross_entropy_loss(probs, target_ids):
    """
    Cross-Entropy Loss 계산
    probs:      (seq_len, vocab_size) - 모델 예측 확률
    target_ids: (seq_len,)            - 정답 인덱스
    """
    eps = 1e-9
    losses = []
    for t, target_id in enumerate(target_ids):
        p_correct = probs[t, target_id]
        loss_t = -np.log(p_correct + eps)
        losses.append(loss_t)
        print(f"  위치 {t}: 정답={idx2word[target_id]:<8} P(정답)={p_correct:.4f}  Loss={loss_t:.4f}")
    total_loss = np.mean(losses)
    return total_loss, losses

# 정답 레이블
print(f"정답 레이블: {tgt_output} → {[idx2word[i] for i in tgt_output]}")
print(f"모델 예측 확률 행렬 형태: {probs.shape}\n")
print("📉 위치별 Loss:")
total_loss, individual_losses = cross_entropy_loss(probs, tgt_output)
print(f"\n✅ 평균 Cross-Entropy Loss: {total_loss:.4f}")
print("\n💡 학습을 통해 이 Loss를 최소화하는 방향으로 가중치를 업데이트합니다 (역전파)!")

# Loss 시각화
fig, ax = plt.subplots(figsize=(7, 4))
tgt_labels = [idx2word[i] for i in tgt_output]
bars = ax.bar(tgt_labels, individual_losses, color=['#EF5350','#42A5F5','#66BB6A'], 
               edgecolor='gray', alpha=0.85)
ax.axhline(total_loss, color='red', linestyle='--', label=f'평균 Loss={total_loss:.4f}')
for bar, loss in zip(bars, individual_losses):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{loss:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_title('위치별 Cross-Entropy Loss', fontsize=13)
ax.set_ylabel('Loss')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 🔄 12. 추론(Inference) - Autoregressive 생성

학습된 모델로 실제 번역을 수행합니다.  
**Teacher Forcing 없이**, 이전 출력을 다음 입력으로 사용합니다.

```
Step 1: [<SOS>]            → 모델 → '나는'
Step 2: [<SOS>, '나는']    → 모델 → '괜찮아'
Step 3: [<SOS>, '나는', '괜찮아'] → 모델 → '<EOS>' → 생성 완료!
```

In [ ]:
def transformer_inference(src_tokens, max_len=10):
    """Transformer Autoregressive 추론"""
    # 인코더 실행
    src_emb = get_embedding(src_tokens)
    src_pe  = positional_encoding(len(src_tokens), d_model)
    enc_in  = src_emb + src_pe
    
    ctx = enc_in.copy()
    for _ in range(N):
        ctx = encoder_layer(ctx)
    
    # 디코더: <SOS>부터 시작
    generated = [vocab['<SOS>']]
    
    print(f"🔄 Autoregressive 생성 과정")
    print("-" * 50)
    
    for step in range(max_len):
        # 현재까지 생성된 시퀀스를 디코더 입력으로
        tgt_emb_cur = get_embedding(generated)
        tgt_pe_cur  = positional_encoding(len(generated), d_model)
        dec_in_cur  = tgt_emb_cur + tgt_pe_cur
        
        # Masked Self-Attention
        mask_cur = create_causal_mask(len(generated))
        msa_out, _ = multi_head_attention(dec_in_cur, dec_in_cur, dec_in_cur,
                                           num_heads=num_heads, mask=mask_cur)
        dec_cur = add_and_norm(dec_in_cur, msa_out)
        
        # Cross-Attention
        ca_out, _ = multi_head_attention(dec_cur, ctx, ctx, num_heads=num_heads)
        dec_cur = add_and_norm(dec_cur, ca_out)
        
        # FFN
        ffn_out, _ = feed_forward(dec_cur)
        dec_cur = add_and_norm(dec_cur, ffn_out)
        
        # 마지막 위치의 출력으로 다음 단어 예측
        last_output = dec_cur[-1:, :]   # (1, d_model)
        logits_cur  = last_output @ W_linear + b_linear  # (1, vocab_size)
        probs_cur   = softmax(logits_cur)[0]              # (vocab_size,)
        
        next_token = np.argmax(probs_cur)
        next_word  = idx2word[next_token]
        
        current_seq = [idx2word[i] for i in generated]
        print(f"  Step {step+1}: 입력={current_seq} → 예측={next_word} (확률={probs_cur[next_token]:.4f})")
        
        if next_token == vocab['<EOS>']:
            print(f"\n  <EOS> 토큰 생성 → 완료!")
            break
        generated.append(next_token)
    
    result = [idx2word[i] for i in generated[1:]]  # <SOS> 제외
    return result

# 추론 실행
print(f"소스 문장: {src_sentence}")
print(f"소스 토큰: {src_tokens}\n")
result = transformer_inference(src_tokens)
print(f"\n✅ 번역 결과: {' '.join(src_sentence)} → {' '.join(result)}")

---
## 🗺️ 13. Transformer 전체 구조 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)

def draw_box(ax, x, y, w, h, text, facecolor, fontsize=10, edgecolor='gray'):
    ax.add_patch(plt.FancyBboxPatch((x, y), w, h,
                                     boxstyle="round,pad=0.1",
                                     facecolor=facecolor, edgecolor=edgecolor,
                                     linewidth=1.5))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='dimgray', lw=1.8))

# ===== 인코더 (왼쪽) =====
enc_x = 0.5
draw_box(ax, enc_x, 0.5, 3.5, 0.7, 'Input: [I, am, fine]', '#E8F5E9', fontsize=10)
draw_arrow(ax, enc_x+1.75, 1.2, enc_x+1.75, 1.5)
draw_box(ax, enc_x, 1.5, 3.5, 0.7, 'Embedding + Position Encoding', '#C8E6C9', fontsize=9)
draw_arrow(ax, enc_x+1.75, 2.2, enc_x+1.75, 2.5)
draw_box(ax, enc_x, 2.5, 3.5, 0.7, 'Multi-Head Self-Attention', '#BBDEFB', fontsize=9)
draw_arrow(ax, enc_x+1.75, 3.2, enc_x+1.75, 3.5)
draw_box(ax, enc_x, 3.5, 3.5, 0.5, 'Add & Norm', '#E3F2FD', fontsize=9)
draw_arrow(ax, enc_x+1.75, 4.0, enc_x+1.75, 4.3)
draw_box(ax, enc_x, 4.3, 3.5, 0.7, 'Feed-Forward Network', '#FFF9C4', fontsize=9)
draw_arrow(ax, enc_x+1.75, 5.0, enc_x+1.75, 5.3)
draw_box(ax, enc_x, 5.3, 3.5, 0.5, 'Add & Norm', '#E3F2FD', fontsize=9)
draw_arrow(ax, enc_x+1.75, 5.8, enc_x+1.75, 6.2)
draw_box(ax, enc_x, 6.2, 3.5, 0.8, 'Encoder Output\n(Context)', '#A5D6A7', fontsize=10, edgecolor='green')
ax.text(enc_x+1.75, 7.5, '🔵 ENCODER', ha='center', fontsize=14, fontweight='bold', color='steelblue')

# ===== 디코더 (오른쪽) =====
dec_x = 5.5
draw_box(ax, dec_x, 0.5, 3.5, 0.7, 'Output Input: [<SOS>, 나는, 괜찮아]', '#FFF3E0', fontsize=9)
draw_arrow(ax, dec_x+1.75, 1.2, dec_x+1.75, 1.5)
draw_box(ax, dec_x, 1.5, 3.5, 0.7, 'Embedding + Position Encoding', '#FFE0B2', fontsize=9)
draw_arrow(ax, dec_x+1.75, 2.2, dec_x+1.75, 2.5)
draw_box(ax, dec_x, 2.5, 3.5, 0.7, 'Masked Multi-Head Self-Attention', '#CE93D8', fontsize=9)
draw_arrow(ax, dec_x+1.75, 3.2, dec_x+1.75, 3.5)
draw_box(ax, dec_x, 3.5, 3.5, 0.5, 'Add & Norm', '#F3E5F5', fontsize=9)
draw_arrow(ax, dec_x+1.75, 4.0, dec_x+1.75, 4.3)
draw_box(ax, dec_x, 4.3, 3.5, 0.7, 'Cross-Attention\n(Q=Decoder, K=V=Encoder)', '#FFCCBC', fontsize=9)
draw_arrow(ax, dec_x+1.75, 5.0, dec_x+1.75, 5.3)
draw_box(ax, dec_x, 5.3, 3.5, 0.5, 'Add & Norm', '#FBE9E7', fontsize=9)
draw_arrow(ax, dec_x+1.75, 5.8, dec_x+1.75, 6.1)
draw_box(ax, dec_x, 6.1, 3.5, 0.7, 'Feed-Forward Network', '#FFF9C4', fontsize=9)
draw_arrow(ax, dec_x+1.75, 6.8, dec_x+1.75, 7.1)
draw_box(ax, dec_x, 7.1, 3.5, 0.7, 'Linear + Softmax', '#FFECB3', fontsize=9)
draw_arrow(ax, dec_x+1.75, 7.8, dec_x+1.75, 8.1)
draw_box(ax, dec_x, 8.1, 3.5, 0.7, 'Output: [나는, 괜찮아, <EOS>]', '#FFE082', fontsize=10, edgecolor='orange')
ax.text(dec_x+1.75, 9.3, '🟠 DECODER', ha='center', fontsize=14, fontweight='bold', color='darkorange')

# Cross-Attention 연결선
ax.annotate('', xy=(dec_x+0.3, 4.65), xytext=(enc_x+3.5, 6.6),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5,
                           connectionstyle='arc3,rad=-0.2'))
ax.text(4.7, 5.5, 'Context\n전달', ha='center', fontsize=10, color='green', fontweight='bold')

ax.set_title('Transformer 전체 아키텍처', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 📝 14. 핵심 개념 정리 & 퀴즈

### ✅ 핵심 개념 요약

| 구성요소 | 역할 | 입력→출력 |
|---------|------|----------|
| **Embedding** | 토큰→벡터 변환 | `(seq_len,)` → `(seq_len, d_model)` |
| **Positional Encoding** | 위치 정보 부여 | `(seq_len, d_model)` |
| **Self-Attention** | 토큰 간 관계 학습 | Q, K, V → `(seq_len, d_v)` |
| **Multi-Head Attention** | 다양한 관점의 Attention | `h`개 Head → Concat → Linear |
| **Add & Norm** | 잔차연결+정규화 | 기울기 소실 방지 |
| **FFN** | 비선형 특징 변환 | `d_model→d_ff→d_model` |
| **Masked Attention** | 미래 정보 차단 | 하삼각 마스크 적용 |
| **Cross-Attention** | 인코더↔디코더 연결 | Q=디코더, K=V=인코더 |
| **Softmax** | 확률 분포 변환 | 로짓→확률 |

---
### 🧠 퀴즈 (직접 생각해보세요!)

In [ ]:
quiz = [
    {
        "q": "Q1. Self-Attention에서 Q, K, V는 각각 무엇을 의미하나요?",
        "hint": "Query=?, Key=?, Value=?",
        "answer": "Q(Query)=질문(무엇을 찾는가), K(Key)=열쇠(어떤 정보를 가졌는가), V(Value)=실제 전달 정보"
    },
    {
        "q": "Q2. Positional Encoding이 필요한 이유는?",
        "hint": "RNN과 달리 트랜스포머는 순서 정보가 없다...",
        "answer": "트랜스포머는 RNN과 달리 순차적으로 처리하지 않아 순서 정보가 없음. PE로 위치 정보를 명시적으로 추가"
    },
    {
        "q": "Q3. Masked Self-Attention에서 마스킹을 하는 이유는?",
        "hint": "학습 시 디코더는 미래 단어를 보면 안 된다...",
        "answer": "디코더는 자기회귀(Autoregressive) 방식으로 생성하므로, 학습 시에도 미래 토큰을 참조하면 안 됨"
    },
    {
        "q": "Q4. Cross-Attention에서 Q, K, V는 각각 어디서 오나요?",
        "hint": "인코더 vs 디코더...",
        "answer": "Q=디코더 출력, K=인코더 출력, V=인코더 출력 (디코더가 인코더 정보를 참조)"
    },
    {
        "q": "Q5. Attention Score를 √d_k로 나누는 이유는?",
        "hint": "차원이 클수록 내적값이 커지면...",
        "answer": "d_k가 크면 내적값이 커져 softmax가 극단적으로 작아짐(기울기 소실). √d_k로 스케일링하여 안정적 학습"
    },
]

print("="*60)
print("🧠 트랜스포머 핵심 개념 퀴즈")
print("="*60)
for i, q in enumerate(quiz):
    print(f"\n{q['q']}")
    print(f"💡 힌트: {q['hint']}")
    print(f"✅ 정답: {q['answer']}")
    print("-"*55)

---
## 🚀 15. PyTorch로 실제 Transformer 구현 (심화)

지금까지 numpy로 이해했다면, 이제 PyTorch 공식 API로 동일한 동작을 확인합니다.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print(f"✅ PyTorch {torch.__version__} 로드 완료")
    PYTORCH_AVAILABLE = True
except ImportError:
    print("❌ PyTorch가 설치되지 않았습니다. pip install torch 를 실행하세요.")
    PYTORCH_AVAILABLE = False

In [ ]:
if PYTORCH_AVAILABLE:
    class SimpleTransformer(nn.Module):
        def __init__(self, vocab_size, d_model=32, nhead=4, 
                     num_encoder_layers=2, num_decoder_layers=2, 
                     dim_feedforward=128, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.vocab_size = vocab_size
            
            # 임베딩
            self.src_embedding = nn.Embedding(vocab_size, d_model)
            self.tgt_embedding = nn.Embedding(vocab_size, d_model)
            
            # Transformer 본체
            self.transformer = nn.Transformer(
                d_model=d_model,
                nhead=nhead,
                num_encoder_layers=num_encoder_layers,
                num_decoder_layers=num_decoder_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True
            )
            
            # 출력 Linear
            self.output_linear = nn.Linear(d_model, vocab_size)
        
        def forward(self, src, tgt, tgt_mask=None):
            # 임베딩 (스케일링)
            src_emb = self.src_embedding(src) * (self.d_model ** 0.5)
            tgt_emb = self.tgt_embedding(tgt) * (self.d_model ** 0.5)
            
            # Transformer
            out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
            
            # 로짓
            logits = self.output_linear(out)
            return logits
    
    # 모델 생성
    model = SimpleTransformer(vocab_size=VOCAB_SIZE, d_model=32, nhead=4)
    
    # 더미 데이터
    src_t = torch.tensor([src_tokens])      # (1, 3)
    tgt_t = torch.tensor([tgt_input])       # (1, 3)
    
    # Causal mask 생성
    tgt_len = tgt_t.shape[1]
    causal_mask_t = nn.Transformer.generate_square_subsequent_mask(tgt_len)
    
    # Forward pass
    with torch.no_grad():
        logits_t = model(src_t, tgt_t, tgt_mask=causal_mask_t)
    
    probs_t = F.softmax(logits_t, dim=-1)
    
    print("🔥 PyTorch Transformer 실행 결과")
    print(f"  입력(src) 형태:  {src_t.shape}")
    print(f"  입력(tgt) 형태:  {tgt_t.shape}")
    print(f"  출력 로짓 형태:  {logits_t.shape}")
    print(f"  출력 확률 형태:  {probs_t.shape}")
    
    # 파라미터 수
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n📊 모델 파라미터 수: {total_params:,}개")
    
    # 예측
    pred_ids = probs_t[0].argmax(dim=-1).tolist()
    pred_words = [idx2word[i] for i in pred_ids]
    print(f"\n📌 각 위치 예측 (학습 전 랜덤): {pred_words}")
    print("(학습 후에는 올바른 번역이 출력됩니다)")
    
    # 모델 구조 출력
    print("\n📐 모델 구조:")
    print(model)
else:
    print("PyTorch를 설치한 후 실행하세요: pip install torch")

---
## 🎓 마무리

### 트랜스포머 핵심 정리

```
Transformer = Encoder + Decoder

Encoder:
  입력 → Embedding + PE
       → Multi-Head Self-Attention + Add&Norm
       → Feed-Forward + Add&Norm
       → Context (× N 반복)

Decoder:
  출력(이전) → Embedding + PE
             → Masked Multi-Head Self-Attention + Add&Norm
             → Cross-Attention(Q=Dec, K=V=Enc) + Add&Norm
             → Feed-Forward + Add&Norm
             → Linear + Softmax
             → 다음 단어 확률 분포 (× N 반복)
```

### 트랜스포머 응용 분야
- **NLP**: GPT, BERT, T5, ChatGPT
- **Vision**: ViT (Vision Transformer)
- **멀티모달**: DALL-E, Flamingo
- **로봇/강화학습**: Decision Transformer
- **생물정보학**: AlphaFold2

---
*강의: 신박AI | [Deep Learning 101] 트랜스포머, 스텝 바이 스텝*  
*실습 노트북 제작: Claude by Anthropic*